# Who Owns the Land? First-attempt map from open data (County Louth)

This notebook pulls together everything that is **already open** and can point to a Catholic Church site in one county, and puts it on one map.

**What it can and can't tell you.** No open dataset in Ireland records who *owns* a parcel. Ownership is only in the Land Registry (folios, paid) and the Registry of Deeds. So every point on this map is a **grade D lead**: a place that looks like a Church site (a church, convent, Catholic-ethos school, religious house), with its location and the record it came from. None is an ownership finding.

What the map *is* good for:
1. **The known-sites list** for pilot step 2: every candidate site, ready to look up on the free Land Registry map to get its folio number.
2. **A measure of what open data covers**: how many sites each source finds, and how many are confirmed by two or more sources.
3. **A seed name list** for pilot step 1, from the Charities Regulator register.

Change `config.COUNTY` to run another county.

## Who has done the most so far

Nobody has published an open, parcel-level map of Church land in Ireland. The closest work:

| Who | What | Scale | Open data? |
| --- | --- | --- | --- |
| Conor Ryan, [Irish Examiner (Oct 2011)](https://www.irishexaminer.com/news/arid-20171915.html) | Went through 1.5 million property parcels (valuation records, every county except Dublin). Found 10,700+ properties owned or occupied by the Church, about 1 in 100 rateable properties. | National, the widest count | No |
| [RTÉ Investigates (2025)](https://presspack.rte.ie/2025/05/01/rte-investigates-christian-brothers-the-assets-the-abusers) | Traced 800+ Christian Brothers properties through trustees named on deeds and folios. | One congregation, the deepest | No |
| Noteworthy and The Journal, [HOLY LAND (2022)](https://www.thejournal.ie/holy-land-5847757-Aug2022/) | Property sales by seven redress orders since 2016, checked against the 2009 asset statements. | Sales only | No |
| [2009 Panel report](https://assets.gov.ie/42460/3a600cd4840b4764afa17c694b73e70d.pdf) | Asset statements of the 18 redress congregations. | Values, not locations | Report only |
| Kevin Cahill, *Who Owns Ireland* (2017) | National land-ownership survey by owner. | National, owner level | No |

Ryan's 2011 method is the one to study first: it suggests the older valuation lists named occupiers, which made a national count possible without folios. The current valuation API does not appear to include occupier names (checked in section 5).

## Data inventory: what exists, where, and what it tells you

| Source | What it gives | Ownership? | Where | Access | In this notebook |
| --- | --- | --- | --- | --- | --- |
| **Land Registry folios** (Tailte Éireann) | Registered owner, burdens, parcel map | **Yes, the ground truth** | [landdirect.ie](https://www.landdirect.ie/) | Paid per search and per folio; free map search for folio numbers; no bulk or open download | No (manual, pilot steps 3–4) |
| **Registry of Deeds** | Deeds since 1708, by grantor and townland | Clues only | Henrietta Street, Dublin | In person, by appointment | No |
| **OpenStreetMap** | Churches, convents, religious land, Catholic-named schools and graveyards, often with outlines | No | [Overpass API](https://overpass-api.de/) | Free, ODbL licence | Yes |
| **NIAH** (National Inventory of Architectural Heritage) | Historic churches, convents, presbyteries, schools, with original and current use | No | [ArcGIS service](https://services-eu1.arcgis.com/HyjXgkV6KGMSF3jt/ArcGIS/rest/services/NIAHBuildingsOpenData/FeatureServer) · [data.gov.ie](https://data.gov.ie/dataset/national-inventory-of-architectural-heritage-niah-national-dataset) | Free | Yes |
| **Department of Education school lists** | Every school with its ethos and coordinates | No (patron ≠ landowner) | [data.gov.ie primary](https://data.gov.ie/dataset/primary-schools) · [post-primary](https://data.gov.ie/dataset/post-primary-schools) | Free; the data.gov.ie copy may be old, the current list is on gov.ie | Yes |
| **Valuation list** (Tailte Éireann) | Rateable properties with use, area and coordinates | No occupier names seen; places of worship are often exempt | [Valuation API](https://tailte.ie/en/valuation/open-data/open-data-information) | Free; filter parameters undocumented | Yes (experimental) |
| **Charities Regulator register** | Legal names of dioceses, parishes, congregations, trusts | Names only, no land | [Register (XLSX)](https://data.gov.ie/dataset/register-of-charities-in-ireland) | Free | Yes (names) |
| **CRO** (Companies Registration Office) | Companies, directors, former names | Names only | [core.cro.ie](https://core.cro.ie/) | Search free; documents and bulk access paid | No |
| **Ryan, McAleese, Mother and Baby Homes reports** | Institutions, operators, dates | No | gov.ie; [Industrial Memories](https://industrialmemories.ucd.ie/) | Free, mostly text, not tables | Hand-built seed list |
| **Townland and county boundaries** | Statutory boundaries | No | [data.gov.ie](https://data.gov.ie/en_GB/dataset/townlands-national-statutory-boundaries-ungeneralised-20241) | Free, CC BY | Not yet |
| **Residential Property Price Register** | Sale prices since 2010 | No | [propertypriceregister.ie](https://www.propertypriceregister.ie/) | Free CSV, no coordinates | Not yet (valuation layer) |
| **INSPIRE cadastral parcels** (Tailte Éireann) | Parcel shapes without owners | No | Tailte Éireann | Check licence | Not yet; worth asking about |

The pattern is clear: **location data is plentiful and open; ownership data is closed and paid.** The project's real work is the bridge between them, which this map sets up but can't cross.

## 1. Setup

Run this notebook from the `first-map` folder on a computer with internet access. Downloads are cached in `data/raw/`, so later runs are fast. Set `REFRESH = True` to download again.

```
pip install -r requirements.txt
jupyter notebook first_map.ipynb
```

In [ ]:
import warnings
import pandas as pd
from IPython.display import display, IFrame

from wotl import build, config, sources

REFRESH = False
config.OUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.simplefilter("always")
print("County:", config.COUNTY)

## 2. OpenStreetMap

Churches tagged Christian (other denominations dropped where tagged), convents and monasteries, land tagged as religious, and schools and graveyards that are tagged or named as Catholic. OSM is volunteer-mapped, so coverage is good for churches and patchy for convents and religious land.

In [ ]:
osm = sources.fetch_osm(REFRESH)
print(len(osm), "points")
display(osm.groupby(["category", "denomination"]).size().rename("points").reset_index())

## 3. NIAH

Historic buildings with a religious or institutional original use. Church of Ireland and other denominations are dropped when the record says so; the rest are kept as `catholic` or `unknown`. NIAH covers buildings of architectural interest only, so newer churches and schools are missing.

In [ ]:
niah = sources.fetch_niah(REFRESH)
print(len(niah), "points")
display(niah.groupby(["category", "denomination"]).size().rename("points").reset_index().sort_values("points", ascending=False).head(20))

## 4. Catholic-ethos schools

The Department of Education lists give each school's ethos. A Catholic ethos means a Catholic patron, which often but not always means Church-owned land (the C&AG estimates 85–87% of the school estate is on land the State doesn't own). If the data.gov.ie copy is out of date, save the current list from gov.ie into `data/raw/` with the same file name and rerun.

In [ ]:
schools = sources.fetch_schools(REFRESH)
print(len(schools), "Catholic-ethos schools in", config.COUNTY)
display(schools[["source", "name", "address"]].head(10))

## 5. Valuation list (experimental)

Rateable properties whose use mentions a church, convent, presbytery, school and so on. Check the columns printed below: if an occupier column exists, it is the closest open thing to ownership and worth a lot. Places of worship are often exempt from rates, so expect few churches here.

In [ ]:
valuation = sources.fetch_valuation(REFRESH)
print(len(valuation), "points")
display(valuation.head(10))

## 6. History seed list

Institutions from the Ryan, McAleese and Mother and Baby Homes reports, entered by hand in `data/seed/history_sites.csv` with their report references. It starts with one verified Louth entry: St Joseph's Industrial School, Dundalk (Sisters of Mercy, Ryan Report vol. II ch. 11). Rows need coordinates to appear on the map.

In [ ]:
history = sources.load_history_sites()
display(history)

## 7. Merge into candidate sites and draw the map

Points from different sources within 100 m of each other become one candidate site. A site found by two or more sources is a stronger lead. Each site gets blank `folio_number` and `registered_owner` columns, to fill in from the free Land Registry map and then the folios.

In [ ]:
frames = [osm, niah, schools, valuation, history]
points = pd.concat([f for f in frames if not f.empty], ignore_index=True) if any(not f.empty for f in frames) else sources._empty()
points, sites = build.merge_sites(points)
county = config.COUNTY.lower()
points.to_csv(config.OUT_DIR / f"{county}_points.csv", index=False)
sites.to_csv(config.OUT_DIR / f"{county}_candidate_sites.csv", index=False)
map_path = config.OUT_DIR / f"{county}_first_map.html"
m = build.draw_map(points, sites, map_path)
print(f"{len(sites)} candidate sites, {int((sites['n_sources'] > 1).sum())} found by 2+ sources")
m

## 8. Coverage: what each source adds

How many candidate sites each source found, and how many it found alone. A source that finds many sites alone is filling gaps; one whose sites are all found elsewhere adds confirmation only.

In [ ]:
if not points.empty:
    per_site = points.groupby("site_id")["source"].agg(lambda s: set(s))
    rows = []
    for src in points["source"].unique():
        has = per_site[per_site.apply(lambda s: src in s)]
        rows.append({"source": src, "sites": len(has), "found only here": int((has.apply(len) == 1).sum())})
    display(pd.DataFrame(rows).sort_values("sites", ascending=False))
display(sites.head(20))

## 9. Seed name list from the Charities Regulator register

Charities whose name sounds Catholic (diocese, parish, sisters, trust, and so on), with a flag for an address in the county. This is a starting point for the pilot's name list, not the list itself: it misses trustees, companies and anything not registered as a charity, and it catches some non-Catholic bodies.

In [ ]:
names = sources.fetch_charity_names(REFRESH)
if not names.empty:
    names.to_csv(config.OUT_DIR / "charity_name_leads.csv", index=False)
    print(len(names), "name leads nationally;", int(names.get("in_county", pd.Series(dtype=bool)).sum()), "with an address in", config.COUNTY)
    display(names[names["in_county"]].head(30) if "in_county" in names else names.head(30))

## 10. What this shows, and what's next

- **Possible now:** a known-sites list and map for Louth from open data, and a seed name list. That covers pilot steps 1 and 2 in draft.
- **Not possible from open data:** who owns any site. That needs the Names Index and folios (pilot steps 3–4).
- **Next:** for each candidate site, look up the folio number on the free Land Registry map and record it in `louth_candidate_sites.csv`. Sites with the same folio collapse into one parcel; sites with no folio go to the Registry of Deeds list.
- **To check:** whether the valuation API exposes occupier names (section 5), and whether Tailte Éireann's cadastral parcel data can be licensed, which would turn points into parcel outlines.